# ZENTRA — person_v2: สอน detector ให้เห็น "คนที่อยู่บนพื้น"

## ปัญหาที่ notebook นี้แก้ (วัดจริงเมื่อ 2026-07-14)

person detector ปัจจุบัน **มองคนที่นอนอยู่กับพื้นไม่เห็นเลย** — ไม่ใช่ conf ต่ำ แต่คือ **0 detections**

| model | คนยืน | **คนนอนกับพื้น** |
|---|---|---|
| `yolo11s` (COCO) | conf 0.90 ✅ | **0 detections** @conf 0.10 |
| `yolo11m` | ✅ | **0 detections** |
| `person_v1.pt` (ของเรา) | conf 0.85 ✅ | **0 detections** |
| `yolo11x` | ✅ | เจอที่ conf 0.24 (ช้าเกินใช้จริง) |

**ทุกโมดูลของ ZENTRA ยืนอยู่บนกล่องคน** → คนที่ล้มแล้วจะ **"ไม่มีตัวตน"** สำหรับทั้งระบบ:
fall ยืนยันไม่ได้ว่าเขายังนอนอยู่ · Zone ไม่เห็นคนหมดสติในพื้นที่อันตราย · PPE ไม่เห็นเขา

**คนที่ต้องการความช่วยเหลือมากที่สุด คือคนที่ระบบทำหาย**

นี่ไม่ใช่ปัญหาขนาดโมเดล — เป็น **ปัญหาข้อมูล** COCO label คน "ยืน" แทบทั้งหมด

---

## วิธีแก้

เทรน `yolo11s` (**ขนาดเดิม → ความเร็วเดิม → ไม่ทำลาย fall cadence**) บนข้อมูลผสม:

- **fall-detection-ca3o8** — 4,497 ภาพ **คลาสเดียว: `Fall-Detected`** (label เฉพาะคนที่ล้มแล้ว) → `person` 
  (เราไม่ได้สอนให้มันแยกแยะการล้ม — Transformer ทำหน้าที่นั้น เราสอนแค่ว่า *"คนที่นอนอยู่ ก็ยังเป็นคน"*)
- **people-detection** — ชุดคนยืนเดิมที่ person_v1 เคยเทรน **ผสมกลับเข้ามากันลืม** 
  (เคยเกิดจริง: person_v1 รุ่นที่เทรนบน coco128 ลืม COCO หมดจนแย่กว่า yolo11s เปล่า ๆ)

## 🔒 กฎเหล็ก

**ห้ามใส่ URFD ลงไปเทรน** — มันคือ**ข้อสอบ**ของ `fall_eval.py` ถ้าเทรนบนมัน ตัวเลข recall ทุกตัวจะกลายเป็นเรื่องโกหก

In [ ]:
!pip -q install ultralytics roboflow

import torch, os
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ไม่มี GPU!')
assert torch.cuda.is_available(), 'เปิด GPU ก่อน: Settings → Accelerator → GPU T4 x2'

# Kaggle: /kaggle/working เท่านั้นที่รอดหลัง session จบ
WORK = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)
print('งานทั้งหมดเซฟที่', WORK)

## 1. ดาวน์โหลดข้อมูล

ใส่ Roboflow API key ของคุณ (Settings → เพิ่มเป็น Kaggle Secret ก็ได้)

> **ใช้ version ที่เป็นภาพดิบ (raw) เท่านั้น** เวอร์ชันที่ Roboflow augment มาแล้วจะมีภาพต้นฉบับเดียวกัน
> กระจายอยู่ทั้ง train และ val → **data leakage** → mAP สวยแต่โกหก (เคยทำให้ PPE รายงาน 0.698 มาแล้ว)

In [ ]:
from roboflow import Roboflow

RF_KEY = ''            # ← ใส่ Roboflow API key
assert RF_KEY, 'ต้องใส่ RF_KEY ก่อน'
rf = Roboflow(api_key=RF_KEY)

# (ก) คนล้ม — standing / falling / fallen พร้อมกล่อง
fall = (rf.workspace('roboflow-universe-projects')
          .project('fall-detection-ca3o8')
          .version(1)                       # v1 = raw images, ไม่ augment
          .download('yolov8', location=f'{WORK}/dl_fall'))

# (ข) คนยืน — ชุดเดิมของ person_v1 (กันลืม)
people = (rf.workspace('krittpass-workspace')
            .project('people-detection-o4rdr-scnrf-m10zy')
            .version(2)
            .download('yolov8', location=f'{WORK}/dl_people'))

!cat {WORK}/dl_fall/data.yaml

## 2. รวม + แบ่ง split แบบไม่รั่ว

`build_person_fallen_dataset.py` ทำ 3 อย่าง:
1. ยุบทุกคลาส (`standing`/`falling`/`fallen`) → `person` คลาสเดียว — **ถ้าเจอคลาสที่ไม่รู้จักจะหยุดทันที** ไม่ทิ้งเงียบ ๆ
2. แบ่ง split ด้วย **hash ของชื่อภาพต้นฉบับ** → ภาพ augment ทุกใบของต้นฉบับเดียวกันตกอยู่ split เดียวกันเสมอ
3. สร้าง **`val_fallen.txt`** — รายชื่อภาพ val ที่มีคนอยู่บนพื้น

In [ ]:
%%writefile build_person_fallen_dataset.py
#!/usr/bin/env python3
"""Build the person dataset that finally includes people who are ON THE FLOOR.

The problem this exists to fix
------------------------------
Measured 2026-07-14 on URFD: a COCO-pretrained person detector — yolo11s, m, and
the fine-tuned person_v1 — returns ZERO detections for a person lying on the floor
who is plainly visible to a human. yolo11l and yolo11x find them only at conf
0.17-0.24, and are far too slow for this CPU.

Every module in ZENTRA stands on the person box. So a worker who falls stops
existing: fall detection cannot confirm they stayed down, the danger zone no
longer sees them, PPE no longer sees them. The person who most needs help is the
one the system loses. That is the bug.

It is not a model-size problem. It is a DATA problem: COCO's `person` annotations
are overwhelmingly upright people, so "human, horizontal" is a shape the detector
has essentially never been shown.

The fix
-------
Fine-tune on people in every posture, and keep showing it upright people at the
same time so it does not trade one for the other.

  fall-detection-ca3o8   4,497 images, ONE class: `Fall-Detected`. It boxes the
                         person who has fallen — and nobody else. Mapped to
                         `person`. We are not teaching the detector to classify a
                         fall (the Transformer does that); we are teaching it that
                         a human lying down is still a human.

  people-detection       the upright set person_v1 was trained on. Mixed back in
                         to prevent catastrophic forgetting. Not hypothetical: an
                         earlier person_v1 trained on coco128 forgot COCO entirely
                         and scored WORSE than stock yolo11s.

⚠️  THE TRAP IN THE FALL SET, AND WHY --autolabel-standing EXISTS
-----------------------------------------------------------------
fall-detection-ca3o8 labels ONLY the fallen person. Any bystander still on their
feet is left unboxed — and YOLO reads an unboxed person as background. Train on
that and you are explicitly teaching the detector to IGNORE people who are
standing up, which is the one thing it currently does well.

This is the exact trap that ruined the PPE dataset (docs/TRAINING_PIPELINE.md:
"a frame with no person box does not mean nobody is in the frame ... feeding it as
background teaches the model to overlook people").

The fix uses each model where it is strong:
  fallen people   → the dataset's human labels   (yolo11s is blind to them)
  standing people → yolo11s at high confidence   (it is excellent at them: 0.90)
A yolo11s box that overlaps no existing label is a person the annotator skipped,
so we add it. Boxes are never REPLACED — human labels always win.

NOT USED, DELIBERATELY: URFD. It is the fall evaluation set. Training the person
detector on it would make every recall number `fall_eval.py` reports a fiction.

Usage
-----
    python scripts/build_person_fallen_dataset.py \
        --fall  <roboflow fall-detection export dir> \
        --people <roboflow people-detection export dir> \
        --out    backend/data/train_dataset/person_v2
"""
from __future__ import annotations

import argparse
import hashlib
import shutil
import sys
from collections import Counter
from pathlib import Path

import yaml

for _s in (sys.stdout, sys.stderr):
    try:
        _s.reconfigure(encoding="utf-8")
    except AttributeError:
        pass

IMG_EXT = {".jpg", ".jpeg", ".png"}

# Every one of these is a person. `fallen` and `falling` are the whole point of
# this dataset; `standing` comes along so the model keeps seeing upright people in
# the same domain. Anything NOT matched here is reported and refused — silently
# dropping a class is how a label space quietly goes wrong.
PERSON_ALIASES = {
    "person", "people", "human", "pedestrian", "worker",
    "standing", "stand", "upright",
    "falling", "fall", "fall-detected", "fall_detected",
    "fallen", "fallen-person", "lying", "lie", "laying", "down",
    "sitting", "sit", "seated", "crouching", "bending",
}
# Which source classes are the ones we are short of. Used only to build a
# FALLEN-ONLY validation split — see why in `main`.
FLOOR_ALIASES = {"falling", "fall", "fall-detected", "fall_detected",
                 "fallen", "fallen-person", "lying", "lie", "laying", "down"}


def _iou(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    iy = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = ix * iy
    ua = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter
    return inter / ua if ua > 0 else 0.0


def _to_xyxy(cx, cy, w, h):
    return (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)


class StandingAutoLabeler:
    """Add the standing people the fall dataset's annotator never boxed.

    fall-detection-ca3o8 boxes the fallen person and nobody else. Every bystander
    left on their feet is, to YOLO, background — so training on those images
    teaches the detector to ignore upright people, destroying the one thing it is
    currently good at.

    yolo11s is the right tool for exactly this and no other: it is excellent on
    upright people (conf 0.90) and completely blind to fallen ones (0 detections,
    measured). So anything it finds at HIGH confidence that does not overlap a
    human label is a standing person the annotator skipped. Add it.

    Human labels are never touched — this only ADDS boxes that nothing covers.
    """

    def __init__(self, conf: float = 0.5, iou_thr: float = 0.3):
        from ultralytics import YOLO
        self.model = YOLO("yolo11s.pt")
        self.conf, self.iou_thr = conf, iou_thr
        self.added = 0

    def extra_boxes(self, img: Path, existing: list[tuple]) -> list[str]:
        r = self.model.predict(str(img), conf=self.conf, classes=[0],
                               verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            return []
        h, w = r.orig_shape
        out = []
        for b in r.boxes.xyxy.tolist():
            n = (b[0] / w, b[1] / h, b[2] / w, b[3] / h)          # → normalized xyxy
            if any(_iou(n, e) > self.iou_thr for e in existing):
                continue                                          # already labelled
            cx, cy = (n[0] + n[2]) / 2, (n[1] + n[3]) / 2
            bw, bh = n[2] - n[0], n[3] - n[1]
            out.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            self.added += 1
        return out


def _norm(name: str) -> str:
    return "".join(ch if ch.isalnum() else "-" for ch in str(name).strip().lower()).strip("-")


def base_name(p: Path) -> str:
    """Group key for a leak-free split.

    Roboflow bakes augmented copies of one photo into several files, all sharing a
    base name before the `.rf.<hash>` suffix. Splitting on the FILE puts copies of
    the same photo in both train and val, and the resulting mAP is a lie — this is
    exactly how the old PPE model came to report 0.698. Split on the PHOTO.
    """
    n = p.name
    return n.split(".rf.")[0] if ".rf." in n else p.stem


def split_of(base: str, val: float, test: float) -> str:
    """Deterministic split from a hash of the photo, not from file order — so a
    re-run, or a different filesystem ordering, produces the exact same split."""
    h = int(hashlib.md5(base.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    if h < test:
        return "test"
    if h < test + val:
        return "val"
    return "train"


def load_names(root: Path) -> dict[int, str]:
    y = root / "data.yaml"
    if not y.exists():
        sys.exit(f"ไม่พบ {y}")
    d = yaml.safe_load(y.read_text(encoding="utf-8"))
    names = d.get("names")
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    return {i: str(n) for i, n in enumerate(names or [])}


def collect(root: Path) -> list[Path]:
    out = []
    for split in ("train", "valid", "val", "test"):
        d = root / split / "images"
        if d.is_dir():
            out += [p for p in d.iterdir() if p.suffix.lower() in IMG_EXT]
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--fall", type=Path, required=True,
                    help="fall-detection-ca3o8 export (standing/falling/fallen)")
    ap.add_argument("--people", type=Path, default=None,
                    help="upright person export — mixed in to prevent forgetting")
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--val", type=float, default=0.15)
    ap.add_argument("--test", type=float, default=0.10)
    ap.add_argument("--autolabel-standing", action="store_true",
                    help="เติมกล่อง 'คนยืน' ที่ dataset คนล้มไม่ได้ label ไว้ (ใช้ yolo11s) "
                         "— ถ้าไม่เติม เท่ากับสอนโมเดลว่าคนยืนคือพื้นหลัง")
    ap.add_argument("--autolabel-conf", type=float, default=0.5)
    args = ap.parse_args()

    sources = [("fall", args.fall)] + ([("people", args.people)] if args.people else [])
    if not args.people:
        print("⚠️  ไม่ได้ใส่ --people → เสี่ยงลืมคนยืน (catastrophic forgetting)\n")

    for _, r in sources:
        if not r.is_dir():
            sys.exit(f"ไม่พบโฟลเดอร์ {r}")

    # ── Resolve every source class to `person`, or refuse ────────────────────
    keep: dict[str, dict[int, bool]] = {}     # source -> {cls_id: is_floor_pose}
    for tag, root in sources:
        names = load_names(root)
        mapping, unknown = {}, []
        for i, n in names.items():
            k = _norm(n)
            if k in PERSON_ALIASES:
                mapping[i] = k in FLOOR_ALIASES
            else:
                unknown.append(n)
        if unknown:
            sys.exit(f"❌ {tag}: ไม่รู้จักคลาส {unknown}\n"
                     f"   เพิ่มลง PERSON_ALIASES (ถ้าเป็นคน) หรือแก้สคริปต์ให้ทิ้งอย่างตั้งใจ\n"
                     f"   — การทิ้งคลาสเงียบ ๆ คือวิธีที่ label space พังโดยไม่มีใครรู้")
        keep[tag] = mapping
        floor = [names[i] for i, f in mapping.items() if f]
        print(f"{tag:7} classes → person : {sorted(names.values())}")
        print(f"{' ':7} ของที่เราขาด     : {floor or '(ไม่มี)'}")

    for split in ("train", "val", "test"):
        for sub in ("images", "labels"):
            (args.out / split / sub).mkdir(parents=True, exist_ok=True)

    stats = Counter()
    floor_val_imgs: list[str] = []      # val images that contain a person ON THE FLOOR

    # Only the fall set needs its standing people recovered; people-detection
    # already boxes everyone.
    auto = StandingAutoLabeler(conf=args.autolabel_conf) if args.autolabel_standing else None
    if auto:
        print("\n🔎 auto-label คนยืนที่ตกหล่นใน fall set (yolo11s, conf "
              f"{args.autolabel_conf}) — อาจใช้เวลาสักครู่\n")

    for tag, root in sources:
        imgs = collect(root)
        print(f"\n{tag}: {len(imgs)} ภาพ")
        for img in imgs:
            lbl = img.parent.parent / "labels" / f"{img.stem}.txt"
            lines, has_floor, boxes = [], False, []
            if lbl.exists():
                for ln in lbl.read_text(encoding="utf-8").splitlines():
                    parts = ln.split()
                    if len(parts) < 5:
                        continue
                    cid = int(float(parts[0]))
                    if cid not in keep[tag]:
                        continue
                    has_floor |= keep[tag][cid]
                    lines.append("0 " + " ".join(parts[1:5]))   # every class → person
                    boxes.append(_to_xyxy(*(float(v) for v in parts[1:5])))
            # Recover the people the annotator left out. Without this, an unboxed
            # bystander is a NEGATIVE example of a person, and we would be training
            # the detector to overlook exactly the upright people it is good at.
            if auto is not None and tag == "fall":
                lines += auto.extra_boxes(img, boxes)
            # An image with genuinely no person is a legitimate background frame and
            # is kept — it teaches what is NOT a person.
            base = base_name(img)
            sp = split_of(f"{tag}/{base}", args.val, args.test)
            name = f"{tag}_{img.stem}"
            shutil.copy2(img, args.out / sp / "images" / f"{name}{img.suffix}")
            (args.out / sp / "labels" / f"{name}.txt").write_text(
                "\n".join(lines), encoding="utf-8")
            stats[f"{sp}_img"] += 1
            stats[f"{sp}_box"] += len(lines)
            if has_floor:
                stats[f"{sp}_floor_img"] += 1
                if sp == "val":
                    floor_val_imgs.append(f"{name}{img.suffix}")

    (args.out / "data.yaml").write_text(yaml.safe_dump({
        "path": str(args.out.resolve()),
        "train": "train/images", "val": "val/images", "test": "test/images",
        "nc": 1, "names": ["person"],
    }, sort_keys=False), encoding="utf-8")

    # A FALLEN-ONLY val list. Overall mAP hides exactly the failure we are fixing:
    # a model can score beautifully by nailing the upright majority while still
    # missing every person on the floor — which is the state person_v1 is in today
    # (great mAP, zero recall on a fallen worker). Score that subset on its own, or
    # you will not know whether this worked.
    (args.out / "val_fallen.txt").write_text("\n".join(floor_val_imgs), encoding="utf-8")

    print("\n" + "=" * 58)
    for sp in ("train", "val", "test"):
        print(f"  {sp:5}  {stats[f'{sp}_img']:6d} ภาพ  {stats[f'{sp}_box']:7d} กล่อง  "
              f"· มีคนอยู่บนพื้น {stats[f'{sp}_floor_img']:5d} ภาพ")
    print("=" * 58)
    if auto is not None:
        print(f"เติมกล่องคนยืนที่ตกหล่น: {auto.added} กล่อง "
              f"(ถ้าไม่เติม กล่องเหล่านี้จะกลายเป็นตัวอย่าง 'พื้นหลัง' ที่สอนให้โมเดลมองข้ามคนยืน)")
    elif any(t == "fall" for t, _ in sources):
        print("⚠️  ไม่ได้ใช้ --autolabel-standing: คนยืนใน fall set ที่ไม่ถูก label\n"
              "    จะถูกเรียนรู้เป็น 'พื้นหลัง' → โมเดลอาจเริ่มมองข้ามคนยืน")
    print(f"data.yaml   → {args.out / 'data.yaml'}")
    print(f"val ที่มีคนล้ม → {args.out / 'val_fallen.txt'}  ({len(floor_val_imgs)} ภาพ)")
    print("\n⚠️  URFD ไม่ได้ถูกใส่เข้ามาโดยเจตนา — มันคือชุดทดสอบของ fall_eval.py")


if __name__ == "__main__":
    main()


In [ ]:
# หมายเหตุ: IPython รัน `!` ทีละบรรทัดเดียว ห้ามตัดบรรทัด — ต้องเป็นคำสั่งเดียวยาว ๆ
#
# --autolabel-standing สำคัญมาก: fall-detection-ca3o8 label เฉพาะ 'คนที่ล้มแล้ว'
# คนยืนที่อยู่ในภาพเดียวกันไม่ถูก label เลย ซึ่ง YOLO จะอ่านว่าเป็น 'พื้นหลัง'
# → เท่ากับสอนโมเดลให้มองข้ามคนยืน ซึ่งเป็นสิ่งเดียวที่มันทำได้ดีอยู่ตอนนี้
# แฟล็กนี้ให้ yolo11s (เก่งคนยืน conf 0.90) เติมกล่องที่ตกหล่นกลับเข้าไป
!python build_person_fallen_dataset.py --fall /kaggle/working/dl_fall --people /kaggle/working/dl_people --out /kaggle/working/person_v2 --autolabel-standing


## 3. เทรน

เริ่มจาก `yolo11s.pt` (COCO) ไม่ใช่ `person_v1.pt` — เพราะข้อมูลที่ใช้ครอบคลุมของเดิมทั้งหมดอยู่แล้ว
และการเริ่มจาก COCO ให้ backbone ที่ยังไม่ over-specialize กับคนยืน

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
model.train(
    data=f'{WORK}/person_v2/data.yaml',
    epochs=100, imgsz=640, batch=32, device=0, workers=4,
    patience=20, cos_lr=True, seed=0, pretrained=True,
    project=WORK, name='person_v2', exist_ok=True,

    # flipud=0.5 — ตั้งใจ และเป็นหัวใจของ notebook นี้
    # คนล้มปรากฏในทุกทิศทาง (หัวซ้าย/หัวขวา/หัวเข้าหากล้อง) การพลิกแนวตั้ง
    # สร้างท่านอนที่หลากหลายขึ้นฟรี ๆ ค่า default ของ ultralytics คือ 0.0
    # เพราะโจทย์ทั่วไปคนไม่กลับหัว — แต่โจทย์เราคือคนกลับหัวโดยเฉพาะ
    flipud=0.5, fliplr=0.5,
    degrees=15.0,          # เอียงกล้อง / ล้มเฉียง
    scale=0.5, translate=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    mosaic=1.0, close_mosaic=10,
    plots=True, val=True,
)
print('best →', f'{WORK}/person_v2/weights/best.pt')

## 4. วัดผล — **แยกวัดเฉพาะ "คนที่อยู่บนพื้น"**

> ตัวเลข mAP รวม **ซ่อนความล้มเหลวที่เรากำลังแก้อยู่พอดี** โมเดลทำคะแนนสวยได้ด้วยการเก่งกับ
> คนยืนซึ่งเป็นคนส่วนใหญ่ ขณะที่ยังพลาดคนล้มทุกคน — ซึ่งคือสภาพของ `person_v1` วันนี้เป๊ะ ๆ
> (mAP ดี แต่ recall บนคนล้ม = 0)
>
> **ตัวเลขที่ต้องดูคือ recall บน val_fallen เท่านั้น** ถ้ามันไม่ขึ้น = ล้มเหลว ไม่ว่า mAP รวมจะสวยแค่ไหน

In [ ]:
import shutil, yaml
from pathlib import Path

W, ROOT = Path(WORK), Path(f'{WORK}/person_v2')

# สร้าง val ย่อยที่มีแต่ภาพ "คนอยู่บนพื้น"
sub = ROOT / 'val_fallen'
for d in ('images', 'labels'):
    (sub / d).mkdir(parents=True, exist_ok=True)
names = (ROOT / 'val_fallen.txt').read_text().split()
for n in names:
    shutil.copy2(ROOT / 'val/images' / n, sub / 'images' / n)
    shutil.copy2(ROOT / 'val/labels' / (Path(n).stem + '.txt'), sub / 'labels' / (Path(n).stem + '.txt'))
(ROOT / 'data_fallen.yaml').write_text(yaml.safe_dump({
    'path': str(ROOT), 'train': 'train/images', 'val': 'val_fallen/images',
    'nc': 1, 'names': ['person']}, sort_keys=False))
print(f'val_fallen: {len(names)} ภาพ')

rows = []
for tag, w in [('yolo11s (COCO, ก่อนเทรน)', 'yolo11s.pt'),
               ('person_v2 (หลังเทรน)', f'{WORK}/person_v2/weights/best.pt')]:
    m = YOLO(w)
    kw = dict(imgsz=640, device=0, verbose=False, plots=False)
    if w == 'yolo11s.pt':
        kw['classes'] = [0]                      # COCO: จำกัดที่คลาส person
    allv = m.val(data=f'{WORK}/person_v2/data.yaml', **kw)
    fal  = m.val(data=str(ROOT / 'data_fallen.yaml'), **kw)
    rows.append((tag, allv.box.map50, allv.box.mr, fal.box.map50, fal.box.mr))

print(f"\n{'':30} {'val ทั้งหมด':>22} {'เฉพาะคนบนพื้น':>24}")
print(f"{'':30} {'mAP50':>10}{'recall':>12} {'mAP50':>11}{'recall':>12}")
for t, a1, a2, f1, f2 in rows:
    print(f'{t:30} {a1:10.3f}{a2:12.3f} {f1:11.3f}{f2:12.3f}')

print('\n👉 ตัวเลขที่ตัดสินคือ recall คอลัมน์ขวาสุด (คนบนพื้น)')
print('   ถ้ามันไม่ขึ้นอย่างมีนัย = notebook นี้ล้มเหลว ไม่ว่า mAP รวมจะสวยแค่ไหน')

## 5. ดูด้วยตา — สำคัญกว่าตัวเลข

ตัวเลขบอกว่า "ดีขึ้น" ได้โดยที่ตายังไม่เชื่อ ดูภาพจริงก่อนเอาไป deploy

In [ ]:
import cv2, matplotlib.pyplot as plt, random

best = YOLO(f'{WORK}/person_v2/weights/best.pt')
old  = YOLO('yolo11s.pt')
picks = random.sample(names, min(4, len(names)))

fig, ax = plt.subplots(2, len(picks), figsize=(5 * len(picks), 8))
for j, n in enumerate(picks):
    img = cv2.imread(str(sub / 'images' / n))
    for i, (m, t, kw) in enumerate([(old, 'yolo11s (เดิม)', {'classes': [0]}),
                                    (best, 'person_v2 (ใหม่)', {})]):
        r = m.predict(img, imgsz=640, conf=0.25, device=0, verbose=False, **kw)[0]
        ax[i, j].imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
        ax[i, j].set_title(f'{t} — เจอ {len(r.boxes)} คน', fontsize=10)
        ax[i, j].axis('off')
plt.tight_layout(); plt.show()

## 6. นำไป deploy

ดาวน์โหลด `person_v2/weights/best.pt` จาก **Output** ของ Kaggle แล้ว:

```bash
# วางไว้ที่
backend/models/person_v2.pt

# แก้ backend/.env
PERSON_MODEL=<abs path>/backend/models/person_v2.pt

# แล้ววัดผลจริง — URFD ไม่เคยถูกเทรน จึงยังเป็นข้อสอบที่สะอาด
python scripts/fall_eval.py --clips backend/data/fall_clips --hold-last 3
```

**เลข baseline ที่ต้องเอาชนะ: recall = 0.000 (0/30)**

> ⚠️ อย่าเปลี่ยนไปใช้ `yolo11m/l/x` แทน — บน CPU ของเครื่อง deploy มันจะเหลือ ~2-4 fps
> ซึ่งยืดหน้าต่าง 30 เฟรมของ fall Transformer จนพัง (ดู `.claude` invariant)